In [33]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier

# Optional models (install if available)
try:
    from xgboost import XGBClassifier
    HAVE_XGB = True
except Exception:
    HAVE_XGB = False

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [34]:
df = pd.read_csv("../Data/weather_daily_2013_2023_after_preprocess.csv")
df

In [35]:
# Build tomorrow's target (shift -1)
df['target_condition'] = df['weather_condition'].shift(-1)
df = df.dropna(subset=['target_condition']).reset_index(drop=True)

# FEATURES: use everything except labels
drop_cols = ['weather_condition', 'target_condition']  # remove labels from features
features = [c for c in df.columns if c not in drop_cols]

X = df[features].copy()
y = df['target_condition'].copy()

# Encode target classes
le = LabelEncoder()
y_enc = le.fit_transform(y)

classes = list(le.classes_)
print("Classes:", classes)
print("Feature count:", len(features))
print("Features:", features)


In [36]:
numeric_features = features  # all are numeric in your case

scaled_preprocessor = ColumnTransformer(
    transformers=[("num", StandardScaler(), numeric_features)],
    remainder="drop"
)

passthrough_preprocessor = ColumnTransformer(
    transformers=[("num", "passthrough", numeric_features)],
    remainder="drop"
)


In [37]:
tscv = TimeSeriesSplit(n_splits=5)

searches = []

# A) Random Forest
rf_pipe = Pipeline([
    ("prep", passthrough_preprocessor),
    ("clf", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced"))
])
rf_params = {
    "clf__n_estimators": [80, 200, 400, 600, 800],
    "clf__max_depth": [None, 4, 6, 10, 14, 18],
    "clf__min_samples_split": [2, 5, 10],
    "clf__min_samples_leaf": [1, 2, 4],
    "clf__max_features": ["sqrt", "log2", None]
}
searches.append(("RandomForest", rf_pipe, rf_params))

# B) Extra Trees (very strong baseline for tabular)
et_pipe = Pipeline([
    ("prep", passthrough_preprocessor),
    ("clf", ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced"))
])
et_params = {
    "clf__n_estimators": [300, 500, 800, 1000],
    "clf__max_depth": [None, 6, 10, 14],
    "clf__min_samples_split": [2, 5, 10],
    "clf__min_samples_leaf": [1, 2, 4],
    "clf__max_features": ["sqrt", "log2", None]
}
searches.append(("ExtraTrees", et_pipe, et_params))

# C) Gradient Boosting
gb_pipe = Pipeline([
    ("prep", passthrough_preprocessor),
    ("clf", GradientBoostingClassifier(random_state=RANDOM_STATE))
])
gb_params = {
    "clf__n_estimators": [150, 250, 400],
    "clf__learning_rate": [0.03, 0.05, 0.1],
    "clf__max_depth": [2, 3, 4],
    "clf__subsample": [0.8, 1.0]
}
searches.append(("GradientBoosting", gb_pipe, gb_params))

# D) Logistic Regression (multinomial)
lr_pipe = Pipeline([
    ("prep", scaled_preprocessor),
    ("clf", LogisticRegression(max_iter=2000, multi_class="multinomial", class_weight="balanced", random_state=RANDOM_STATE))
])
lr_params = {
    "clf__C": np.logspace(-3, 2, 20),
    "clf__penalty": ["l2"],
    "clf__solver": ["lbfgs", "saga"]
}
searches.append(("LogisticRegression", lr_pipe, lr_params))

# E) SVC (can be strong; use class_weight balanced)
svc_pipe = Pipeline([
    ("prep", scaled_preprocessor),
    ("clf", SVC(probability=True, class_weight="balanced", random_state=RANDOM_STATE))
])
svc_params = {
    "clf__C": np.logspace(-2, 2, 15),
    "clf__gamma": ["scale", "auto"],
    "clf__kernel": ["rbf"]   # rbf is usually strongest
}
searches.append(("SVC", svc_pipe, svc_params))

# F) MLP (Neural net on tabular)
mlp_pipe = Pipeline([
    ("prep", scaled_preprocessor),
    ("clf", MLPClassifier(max_iter=500, random_state=RANDOM_STATE))
])
mlp_params = {
    "clf__hidden_layer_sizes": [(64,), (128,), (64,32), (128,64)],
    "clf__alpha": [1e-5, 1e-4, 1e-3],
    "clf__learning_rate_init": [1e-3, 3e-3, 1e-2],
    "clf__activation": ["relu", "tanh"]
}
searches.append(("MLP", mlp_pipe, mlp_params))

# G) XGBoost (if available)
if HAVE_XGB:
    xgb_pipe = Pipeline([
        ("prep", passthrough_preprocessor),
        ("clf", XGBClassifier(
            objective="multi:softprob",
            eval_metric="mlogloss",
            tree_method="hist",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])
    xgb_params = {
        "clf__n_estimators": [300, 500, 800],
        "clf__max_depth": [3, 4, 5, 6],
        "clf__learning_rate": [0.03, 0.05, 0.1],
        "clf__subsample": [0.7, 0.9, 1.0],
        "clf__colsample_bytree": [0.7, 0.9, 1.0],
        "clf__min_child_weight": [1, 3, 5]
    }
    searches.append(("XGBoost", xgb_pipe, xgb_params))


In [38]:
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y_enc[:split_idx], y_enc[split_idx:]

In [39]:
RF = RandomForestClassifier(n_estimators=80, max_depth= 4, random_state=0)
RF = RF.fit(X_train, y_train)

In [40]:
y_pred = RF.predict(X_test)

In [41]:
from sklearn.metrics import accuracy_score
score = accuracy_score(y_test, y_pred) * 100
print("Accuracy using Random Forest Classifier: ", round(score, 1), "%" )

In [20]:
results = []

for name, pipe, param_grid in searches:
    print(f"\n=== Tuning {name} ===")
    search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=param_grid,
        n_iter=30,                      # adjust if you want deeper search
        scoring="f1_macro",
        cv=tscv,
        n_jobs=-1,
        verbose=1,
        random_state=RANDOM_STATE
    )
    search.fit(X_train, y_train)
    
    best_model = search.best_estimator_
    y_pred = best_model.predict(X_test)
    macro_f1 = f1_score(y_test, y_pred, average="macro")
    acc = accuracy_score(y_test, y_pred)

    print(f"\nBest params for {name}: {search.best_params_}")
    print(f"Validation (hold-out) Macro-F1: {macro_f1:.4f} | Accuracy: {acc:.4f}")
    print("\nClassification report (hold-out):")
    print(classification_report(y_test, y_pred, target_names=classes, digits=4))

    results.append({
        "model": name,
        "best_estimator": best_model,
        "best_params": search.best_params_,
        "macro_f1": macro_f1,
        "accuracy": acc
    })

# Rank models by macro-F1
results_sorted = sorted(results, key=lambda d: d["macro_f1"], reverse=True)
print("\n=== Model ranking (by Macro-F1 on hold-out) ===")
for r in results_sorted:
    print(f"{r['model']}: Macro-F1={r['macro_f1']:.4f}, Acc={r['accuracy']:.4f}")
